# Metal (surgical pin) detection check — why `bmd_metal_ratio` fails

**Purpose.** The `bmd-viewer` backend excludes metal implants from the BMD measurement with

```
metal_thr = bmd_metal_ratio * p95(body)      # bmd_metal_ratio = 1.3 by default
valid &= dens < metal_thr
```

This rule was tuned on **synthetic blobs**. This notebook validates it on **real radiographs**
of post-operative patients and shows, with images, why it does not work.

**Cohorts** (`etc/dataset/dataset-dcm/`)
- `lateral`      — 176 studies, assumed **no metal**
- `lateral-pin`  —   9 studies, **pedicle screws / pins confirmed visually**

> **Result (spoiler).** The rule is a **no-op**: in 8 of 9 metal studies the threshold
> `1.3 x p95` is **larger than the image maximum**, so no pixel can ever be flagged. Worse, the
> brightest pixels are **not** the pin, so *no* intensity threshold can isolate it. The metal must
> be found by **shape/edges**, or from `FOR PROCESSING` originals where linearity survives.

## 0. Imports & configuration

In [ ]:
import glob, os, sys
import numpy as np
import cv2
import matplotlib.pyplot as plt

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")   # Windows console
except Exception:
    pass

BASE     = r"C:\Users\csm02\Desktop\edward\bmd\etc\dataset\dataset-dcm"
PIN_DIR  = os.path.join(BASE, "lateral-pin")     # metal present
NOPIN_DIR = os.path.join(BASE, "lateral")        # metal absent
OUT      = r"C:\Users\csm02\Desktop\edward\bmd\src\BMD\model\metal_check"
os.makedirs(OUT, exist_ok=True)

METAL_RATIO = 1.3      # backend settings.bmd_metal_ratio
TOP_FRAC    = 0.001    # top 0.1% brightest
DICOM_EXTS  = (".dcm", ".dicom")   # this cohort uses ".dicom"

def dicom_files(d):
    out = []
    for e in DICOM_EXTS:
        out += glob.glob(os.path.join(d, "*" + e)) + glob.glob(os.path.join(d, "*" + e.upper()))
    return sorted(set(out))

print("pin :", len(dicom_files(PIN_DIR)), "| no-pin:", len(dicom_files(NOPIN_DIR)))

## 1. Loaders — identical to the backend

`dens` reproduces `YoloBmdEngine._load_dicom()` exactly (rescale + MONOCHROME1 inversion so that
bone is bright), and `body_mask` reproduces the Otsu background rejection of
`_valid_tissue_mask()`. Keeping these identical is what makes the conclusion transferable.

In [ ]:
TAGS = ["PhotometricInterpretation", "PixelIntensityRelationship", "BitsStored",
        "RescaleType", "RescaleSlope", "RescaleIntercept", "WindowCenter",
        "WindowWidth", "VOILUTFunction", "PresentationLUTShape", "Modality"]

def load(path):
    """-> (dens float32 'bright bone', header dict). Mirrors the backend loader."""
    import pydicom
    ds  = pydicom.dcmread(path)
    raw = ds.pixel_array.astype(np.float32)
    raw = raw * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    mono1 = str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1"
    if mono1:
        raw = raw.max() - raw
    meta = {t: str(getattr(ds, t, "")) for t in TAGS}
    meta["_mono1_inverted"] = str(mono1)
    return raw, meta

def body_mask(dens):
    """Otsu foreground (body) mask - same background rejection as _valid_tissue_mask()."""
    u8 = cv2.normalize(dens, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    thr, _ = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return u8 >= thr

def disp(dens):
    """0.5-99.5 percentile display window (what the eye sees)."""
    lo, hi = np.percentile(dens, [0.5, 99.5])
    if hi <= lo:
        lo, hi = float(dens.min()), float(dens.max()) + 1e-6
    return np.clip((dens - lo) / (hi - lo), 0, 1)

print("loaders ready")

## 2. Diagnostic overlay — where is the metal, in pixel-value terms?

Four panels per study:
1. **Original** in the display window (the pin is obvious to the eye),
2. **Top 0.1% brightest** pixels (cyan) — if the pin were the densest object it would light up here,
3. **What the current rule flags** (red) at `1.3 x p95`,
4. **Body histogram** with `p95`, `1.3 x p95` and `max` marked — this is where the failure becomes
   undeniable: the red line often sits *beyond* the maximum.

In [ ]:
def diagnose(path, save=True, show=False):
    name = os.path.splitext(os.path.basename(path))[0]
    dens, meta = load(path)
    bm   = body_mask(dens)
    body = dens[bm]
    p95  = float(np.percentile(body, 95))
    mx   = float(body.max())
    thr_metal = METAL_RATIO * p95
    flagged   = bm & (dens >= thr_metal)
    top_cut   = float(np.quantile(body, 1.0 - TOP_FRAC))
    top       = bm & (dens >= top_cut)

    n_lab, lab, cc, _ = cv2.connectedComponentsWithStats(top.astype(np.uint8), 8)
    big = int(cc[1:, cv2.CC_STAT_AREA].max()) if n_lab > 1 else 0

    d   = disp(dens)
    rgb = cv2.cvtColor((d * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    v_top = rgb.copy(); v_top[top] = (0, 255, 255)
    v_fl  = rgb.copy(); v_fl[flagged] = (255, 0, 0)

    fig, ax = plt.subplots(1, 4, figsize=(20, 5.6))
    ax[0].imshow(d, cmap="gray"); ax[0].set_title("1) Original (0.5-99.5% window)", fontsize=10)
    ax[1].imshow(v_top); ax[1].set_title(f"2) Top {TOP_FRAC*100:.1f}% brightest (cyan)\n"
                                         f"largest blob = {big} px", fontsize=10)
    ax[2].imshow(v_fl);  ax[2].set_title(f"3) Current rule: dens >= {METAL_RATIO}x p95 (red)\n"
                                         f"flagged = {int(flagged.sum())} px", fontsize=10)
    for a in ax[:3]:
        a.axis("off")
    ax[3].hist(body.ravel(), bins=120, color="#777777"); ax[3].set_yscale("log")
    ax[3].axvline(p95,       color="dodgerblue", ls="--", label=f"p95 = {p95:.0f}")
    ax[3].axvline(thr_metal, color="red",        ls="-",  label=f"{METAL_RATIO}x p95 = {thr_metal:.0f}")
    ax[3].axvline(mx,        color="k",          ls=":",  label=f"max = {mx:.0f}")
    ax[3].set_title(f"4) Body histogram (log)\nmax/p95 = {mx/p95:.3f}", fontsize=10)
    ax[3].legend(fontsize=8)
    unreachable = " -- THRESHOLD ABOVE MAX (rule can never fire)" if thr_metal > mx else ""
    fig.suptitle(f"{name[:24]}   flagged={int(flagged.sum())} px{unreachable}", fontsize=12)
    plt.tight_layout()
    if save:
        fig.savefig(os.path.join(OUT, f"{name[:16]}.png"), dpi=110)
    (plt.show() if show else plt.close(fig))
    return {"name": name, "p95": p95, "max": mx, "max_over_p95": mx / p95,
            "thr": thr_metal, "flagged_px": int(flagged.sum()),
            "unreachable": thr_metal > mx, "top_blob_px": big, "meta": meta}

rows = [diagnose(f, save=True, show=(i < 2)) for i, f in enumerate(dicom_files(PIN_DIR))]

print(f"\n{'study':18s} {'p95':>8s} {'max':>8s} {'1.3xp95':>9s} {'max/p95':>8s} {'flagged':>8s}  note")
for r in rows:
    print(f"{r['name'][:18]:18s} {r['p95']:8.0f} {r['max']:8.0f} {r['thr']:9.0f} "
          f"{r['max_over_p95']:8.3f} {r['flagged_px']:8d}  "
          f"{'UNREACHABLE' if r['unreachable'] else ''}")
n_un = sum(r["unreachable"] for r in rows)
print(f"\n=> threshold unreachable in {n_un}/{len(rows)} metal studies "
      f"=> the metal exclusion is a NO-OP on this cohort.")

## 3. Why intensity cannot work here — the header explains it

`MONOCHROME2` + `WindowCenter`/`WindowWidth` present + `BitsStored = 12` with `max = 4095`
(a clipped 12-bit ceiling) mean these are **presentation ("for processed") images**: a display LUT
has already compressed the highlights, so the extreme attenuation of metal has been squashed down
to the same range as dense bone. The ratio-to-`p95` heuristic assumes **linear** attenuation, which
this data no longer has.

In [ ]:
print("=== DICOM header across the metal cohort ===")
for t in TAGS + ["_mono1_inverted"]:
    vals = sorted({r["meta"].get(t, "") for r in rows})
    if any(vals):
        print(f"  {t:28s}: {vals[:4]}")

print("\n=== ratio sweep: no threshold separates the cohorts ===")
def flagged_frac(path, ratio):
    dens, _ = load(path); bm = body_mask(dens); body = dens[bm]
    return float((body >= ratio * np.percentile(body, 95)).mean())

pin   = dicom_files(PIN_DIR)
nopin = dicom_files(NOPIN_DIR)[:40]          # sample for speed
print(f"{'ratio':>6s} | {'metal img%':>11s} | {'no-metal img%':>14s}")
for ratio in [1.05, 1.1, 1.2, 1.3, 1.5, 2.0]:
    a = 100 * np.mean([flagged_frac(f, ratio) > 1e-5 for f in pin])
    b = 100 * np.mean([flagged_frac(f, ratio) > 1e-5 for f in nopin])
    print(f"{ratio:6.2f} | {a:10.1f}% | {b:13.1f}%")
print("\n(metal% should be ~100 and no-metal% ~0. It never is -> intensity alone cannot separate.)")

## 4. The right approach — find metal by **shape**, not brightness

The pin is trivially visible to a human because of its **geometry**, not its grey level: perfectly
straight edges, a uniform interior, and a very sharp boundary against bone. The detector below uses
that instead of intensity:

1. strong **gradient magnitude** (Scharr) — metal edges are far sharper than bone trabeculae,
2. **morphological closing** to fill the rod/screw interior,
3. keep only **large, solid, elongated** components (`solidity`, `extent`, area).

> ⚠️ **Experimental — first attempt, not yet validated.** Tune on the 9 metal studies *and* verify
> it fires on **none** of the 176 metal-free ones before wiring it into the backend: a false
> positive would silently delete real bone from the measurement.

In [ ]:
def detect_metal_by_shape(dens, grad_pct=99.0, min_area=300):
    """Experimental: locate metal hardware by edge sharpness + shape, not brightness.

    Returns a boolean mask. Intended to REPLACE the intensity rule for presentation images.
    """
    bm = body_mask(dens)
    u8 = cv2.normalize(dens, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    u8 = cv2.GaussianBlur(u8, (3, 3), 0)

    gx = cv2.Scharr(u8, cv2.CV_32F, 1, 0)
    gy = cv2.Scharr(u8, cv2.CV_32F, 0, 1)
    mag = cv2.magnitude(gx, gy)
    mag[~bm] = 0

    thr = np.percentile(mag[bm], grad_pct)          # only the sharpest edges in the body
    edges = ((mag >= thr) & bm).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    solid = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, k, iterations=2)
    solid = cv2.morphologyEx(solid, cv2.MORPH_OPEN,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))

    n, lab, stats, _ = cv2.connectedComponentsWithStats(solid, 8)
    out = np.zeros_like(solid, bool)
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        w, h = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        extent = area / float(w * h + 1e-6)          # how completely it fills its bbox
        comp = (lab == i).astype(np.uint8)
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        hull_a = cv2.contourArea(cv2.convexHull(cnts[0])) if cnts else 0
        solidity = area / (hull_a + 1e-6)            # metal is convex & solid; bone texture is not
        if extent > 0.35 and solidity > 0.80:
            out |= comp.astype(bool)
    return out

# --- eyeball it on the metal cohort ---
files = dicom_files(PIN_DIR)[:4]
fig, axes = plt.subplots(2, len(files), figsize=(4.4 * len(files), 9))
axes = np.atleast_2d(axes)
for j, f in enumerate(files):
    dens, _ = load(f)
    m = detect_metal_by_shape(dens)
    d = disp(dens)
    rgb = cv2.cvtColor((d * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    vis = rgb.copy(); vis[m] = (255, 0, 0)
    axes[0, j].imshow(d, cmap="gray"); axes[0, j].set_title(os.path.basename(f)[:14], fontsize=9)
    axes[1, j].imshow(vis); axes[1, j].set_title(f"shape-based metal: {int(m.sum())} px", fontsize=9)
    axes[0, j].axis("off"); axes[1, j].axis("off")
fig.suptitle("Experimental shape-based metal detection (red) - VERIFY before trusting", fontsize=13)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "shape_based_detection.png"), dpi=110); plt.show()

print("False-positive check on metal-FREE studies (should all be ~0 px):")
for f in dicom_files(NOPIN_DIR)[:8]:
    dens, _ = load(f)
    print(f"  {os.path.basename(f)[:16]}: {int(detect_metal_by_shape(dens).sum()):6d} px")

## 5. Conclusions

1. **The shipped rule is a no-op on real data.** In 8 of 9 metal studies `1.3 x p95` exceeds the
   image maximum, so the metal exclusion can never fire. It was only ever exercised by synthetic
   blobs, which preserved the linear brightness relationship that real presentation images do not.
2. **Re-tuning `bmd_metal_ratio` cannot fix it.** The two cohorts overlap at every ratio, and the
   brightest pixels in a metal study are typically *not* the pin (often a marker or collimation
   edge at the image border). Lowering the ratio flags dense bone before it flags metal.
3. **Impact on the viewer is currently small, by luck of anatomy.** The implants are at the hip
   while BMD is measured at **L4**, so metal rarely overlaps the measurement ROI. The real risk is
   metal distorting the *global* anchors — and the images where a pin sits near the lumbar spine
   (as in this cohort) are exactly the ones where the rule silently does nothing.
4. **Fix, in order of preference:**
   - use the **`FOR PROCESSING`** original DICOM if the PACS still has it — linearity is intact and
     the existing rule works as designed;
   - otherwise switch to **shape/edge-based** detection (Section 4) and validate it on all 176
     metal-free studies before enabling it.

**Do not enable the experimental detector in the backend until the false-positive check in
Section 4 is clean** — wrongly deleting bone from the ROI is worse than not excluding metal.